# 14 — Alineación del Experimento A: ámbito y rol de decisiones con varios hablantes

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [`12-jss-extraccion-openkos-es2015.ipynb`](12-jss-extraccion-openkos-es2015.ipynb) ·
[protocolo de alineación](../docs/protocols/alignment.md) ·
[ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md) ·
[issue #99](https://github.com/jasonssdev/mcd-afg/issues/99)

---

> **Qué es esto y qué no es.** Es un análisis de **insumos** para dos decisiones metodológicas
> del puntaje del Experimento A (issue #99): (1) si una decisión extraída puede alinearse con
> cualquier decisión de la serie o solo con las de su misma reunión, y (2) qué rol de hablante
> se asigna a una decisión de referencia cuya evidencia reúne a varias personas. También revisa
> si los intervalos de confianza, que tratan cada decisión como independiente, cambian al
> considerar que las decisiones vienen de cuatro reuniones. **No decide nada**: la decisión es
> de jss y se registra en un ADR aparte.
>
> **Límites que hay que tener presentes.** ES2015 es una serie de **desarrollo**: ninguna cifra
> de este notebook entra en una tabla de resultados ([ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md)).
> La versión de OpenKOS (0.5.5) es **provisional**. Con una sola serie, cuatro reuniones, 27
> decisiones de referencia y tres corridas de una misma configuración, la muestra es pequeña:
> las cifras indican una dirección y no un tamaño de efecto.

---

# 1. Qué se compara

El puntaje (`src/afg/evaluation/scoring.py`) empareja cada decisión extraída con, a lo más, una
decisión de la referencia adjudicada, por similitud de embeddings (`bge-m3`) del par
(objeto, contenido) y asignación uno a uno (algoritmo húngaro), con el umbral τ. Dos variantes
del **ámbito** de ese emparejamiento:

- **Misma reunión** (`alignment_scope="origin_meetings"`, el valor primario en el código): una
  decisión extraída solo puede emparejarse con decisiones de referencia de la reunión de la que
  proviene. Es la variante estricta. En estas extracciones ninguna decisión trae más de una
  reunión de origen, de modo que «reuniones de origen» equivale a «su reunión».
- **Toda la serie** (`alignment_scope="series"`): puede emparejarse con cualquier decisión de la
  referencia de la serie. Es la alternativa más permisiva.

Y dos formas de asignar el **rol** a una decisión de referencia cuya evidencia reúne a varios
roles (en ES2015, 9 de 27 según el issue; se verifica abajo): el rol **mayoritario** (empate:
el turno más temprano) o **contarla en cada rol** involucrado.

Las extracciones son las de la carpeta `es2015-explora-r3x` (semillas 42, 43 y 44). La corrida
de `es2015-explora-r1` se omite: es idéntica a la corrida 0 de `r3x` (misma semilla,
[notebook 12](12-jss-extraccion-openkos-es2015.ipynb), sección 5) y contarla sería duplicar.

Las funciones nuevas de este análisis están en `src/afg/evaluation/alignment_analysis.py`
(pares emparejados, similitud coseno e intervalo por conglomerados de reuniones); el resto se
reutiliza de `scoring.py`.

In [1]:
import numpy as np
import pandas as pd

from afg.conditions.c1_retrieval import OllamaEmbedder
from afg.domain.decision import Decision
from afg.evaluation.alignment_analysis import (
    embedding_similarity,
    meeting_cluster_intervals,
    scored_matches,
)
from afg.evaluation.reference import load_series_reference
from afg.evaluation.scoring import (
    evidence_role_resolver,
    evidence_roles_resolver,
    interval_settings_from_config,
    pool_scores,
    score_series,
)
from afg.experiments.extraction_runs import load_extraction_runs
from afg.shared.config import load_experiments_config
from afg.shared.paths import AMI_DIR, EXPERIMENT_A_DIR, TRANSCRIPTS_DIR

pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 110)

RUN_DIR = EXPERIMENT_A_DIR / "es2015-explora-r3x"
SERIES = "ES2015"
SCOPES = {"misma reunión": "origin_meetings", "toda la serie": "series"}
TAU_GRID = load_experiments_config()["alignment"]["tau_grid"]
DETAIL_TAUS = (0.50, 0.60, 0.70)

# Guard: extractions, AMI summlink layer and frozen transcripts must be present.
for path in (RUN_DIR, AMI_DIR, TRANSCRIPTS_DIR):
    if not path.exists():
        raise SystemExit(f"No se encontro {path}.\nVer notebooks/README.md (Reproducibilidad).")

runs = load_extraction_runs([RUN_DIR])
decisions_by_run = [[Decision(**d) for d in r.decisions] for r in runs]
reference_set = load_series_reference(SERIES)
reference = list(reference_set.decisions)
settings = interval_settings_from_config()

# Embeddings through the local Ollama model configured for alignment (cached per text).
_embedder = OllamaEmbedder.from_config()
_cache: dict[str, list[float]] = {}


def embed(texts):
    missing = [t for t in dict.fromkeys(texts) if t not in _cache]
    if missing:
        _cache.update(zip(missing, _embedder.embed(missing), strict=True))
    return [_cache[t] for t in texts]


similarity = embedding_similarity(embed)

print(f"Corridas: {[r.label for r in runs]}")
print(f"Decisiones extraídas por corrida: {[len(d) for d in decisions_by_run]}")
print(f"Decisiones de referencia: {len(reference)}; modelo de embeddings: {_embedder.model}")
print(
    f"Con alguna decisión extraída de varias reuniones de origen: "
    f"{sum(bool(d.origin_meeting_ids) for ds in decisions_by_run for d in ds)}"
)
print(f"Grilla de τ configurada: {TAU_GRID}")

Corridas: ['es2015-explora-r3x / corrida 0', 'es2015-explora-r3x / corrida 1', 'es2015-explora-r3x / corrida 2']
Decisiones extraídas por corrida: [71, 69, 69]
Decisiones de referencia: 27; modelo de embeddings: bge-m3
Con alguna decisión extraída de varias reuniones de origen: 0
Grilla de τ configurada: [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]


---

# 2. Ámbito: misma reunión frente a toda la serie

Primero, las dos variantes a lo largo de **toda la grilla de τ** configurada: el protocolo
exige mirar una curva y no un punto, y no hay un τ único fijado todavía. Las cifras son las
**agrupadas** (micro) de las tres corridas: se suman los aciertos, los falsos positivos y los
omitidos de cada corrida. TP son los emparejamientos; FP, las extraídas sin pareja; FN, las de
referencia sin pareja.

In [2]:
def run_scores(tau, scope, **kwargs):
    """One SeriesScore per run (accepted decisions only)."""
    return [
        score_series(
            SERIES, reference, decisions, similarity, tau=tau, alignment_scope=scope, **kwargs
        )
        for decisions in decisions_by_run
    ]


def pooled(tau, scope, **kwargs):
    return pool_scores(
        run_scores(tau, scope, **kwargs), settings=settings, partition_of=lambda s: "development"
    )


rows = []
for tau in TAU_GRID:
    for label, scope in SCOPES.items():
        p = pooled(tau, scope)
        rows.append(
            {
                "τ": tau,
                "ámbito": label,
                "TP": p.tp,
                "FP": p.fp,
                "FN": p.fn,
                "precisión": p.precision,
                "cobertura": p.recall,
                "F1": p.f1,
            }
        )
grid = pd.DataFrame(rows)
grid.pivot(index="τ", columns="ámbito", values=["TP", "precisión", "cobertura", "F1"]).round(3)

TP                   precisión                   cobertura                          F1              
ámbito misma reunión toda la serie misma reunión toda la serie misma reunión toda la serie misma reunión toda la serie
τ                                                                                                                     
0.50            63.0          81.0         0.301         0.388         0.778         1.000         0.434         0.559
0.55            54.0          69.0         0.258         0.330         0.667         0.852         0.372         0.476
0.60            42.0          62.0         0.201         0.297         0.519         0.765         0.290         0.428
0.65            23.0          36.0         0.110         0.172         0.284         0.444         0.159         0.248
0.70            10.0          13.0         0.048         0.062         0.123         0.160         0.069         0.090
0.75             3.0           3.0         0.014         0.014         0.037         0.037         0.021         0.021
0.80             0.0           0.0         0.000         0.000         0.000         0.000         0.000         0.000
0.85             0.0           0.0         0.000         0.000         0.000         0.000         0.000         0.000
0.90             0.0           0.0         0.000         0.000         0.000         0.000         0.000         0.000

**Resultado.** La variante de toda la serie da cifras mayores o iguales que la de la misma reunión
en **todos** los τ: el orden entre ámbitos no cambia a lo largo de la grilla, solo cambia la
distancia entre ellos. Con τ = 0,50, toda la serie empareja las 27 decisiones de referencia en
cada corrida (cobertura 1,00) frente a 0,78 en la misma reunión; con τ = 0,60, F1 agrupado
es 0,43 frente a 0,29; con τ = 0,70, 0,09 frente a 0,07; desde τ = 0,80 ambas valen cero, porque
ningún par de (objeto, contenido) alcanza esa similitud (la extracción está en inglés y la
referencia en español). Dos lecturas: (a) el ámbito mueve las cifras con τ bajos, donde los
emparejamientos son holgados, y casi no las mueve con τ altos; (b) la **elección de τ pesa mucho
más que la del ámbito**: dentro de un mismo ámbito, F1 pasa de 0,43 a 0,07 entre τ = 0,50 y
0,70. La precisión es baja en ambos ámbitos (de 0,30 a 0,39 en el mejor caso), coherente con
el exceso de decisiones extraídas del [notebook 12](12-jss-extraccion-openkos-es2015.ipynb).

## 2.1 Cifras con intervalos, por corrida y agrupadas

Para tres valores de τ del tramo en que las cifras no son todo ceros (**ninguno es el elegido**:
el τ de la tesis no está fijado y este análisis no lo fija). Precisión y cobertura llevan el
intervalo de Wilson y F1 el *bootstrap* por decisión, (los intervalos de referencia de `pool_scores`, `*_interval_reference`)
(95 %). «Agrupado» suma las tres corridas.

In [3]:
def fmt(value, interval):
    return f"{value:.2f} [{interval[0]:.2f}, {interval[1]:.2f}]"


rows = []
for tau in DETAIL_TAUS:
    for label, scope in SCOPES.items():
        scores = run_scores(tau, scope)
        items = [(f"corrida {i}", s) for i, s in enumerate(scores)] + [("agrupado", None)]
        for name, s in items:
            p = (
                pool_scores([s], settings=settings, partition_of=lambda x: "development")
                if s
                else pooled(tau, scope)
            )
            rows.append(
                {
                    "τ": tau,
                    "ámbito": label,
                    "corrida": name,
                    "TP": p.tp,
                    "precisión": fmt(p.precision, p.precision_interval_reference),
                    "cobertura": fmt(p.recall, p.recall_interval_reference),
                    "F1": fmt(p.f1, p.f1_interval_reference),
                }
            )
pd.DataFrame(rows)

,τ,ámbito,corrida,TP,precisión,cobertura,F1
0,0.5,misma reunión,corrida 0,21,"0.30 [0.20, 0.41]","0.78 [0.59, 0.89]","0.43 [0.31, 0.54]"
1,0.5,misma reunión,corrida 1,21,"0.30 [0.21, 0.42]","0.78 [0.59, 0.89]","0.44 [0.32, 0.55]"
2,0.5,misma reunión,corrida 2,21,"0.30 [0.21, 0.42]","0.78 [0.59, 0.89]","0.44 [0.31, 0.54]"
3,0.5,misma reunión,agrupado,63,"0.30 [0.24, 0.37]","0.78 [0.68, 0.85]","0.43 [0.37, 0.50]"
4,0.5,toda la serie,corrida 0,27,"0.38 [0.28, 0.50]","1.00 [0.88, 1.00]","0.55 [0.42, 0.66]"
5,0.5,toda la serie,corrida 1,27,"0.39 [0.28, 0.51]","1.00 [0.88, 1.00]","0.56 [0.43, 0.67]"
6,0.5,toda la serie,corrida 2,27,"0.39 [0.28, 0.51]","1.00 [0.88, 1.00]","0.56 [0.43, 0.67]"
7,0.5,toda la serie,agrupado,81,"0.39 [0.32, 0.46]","1.00 [0.95, 1.00]","0.56 [0.49, 0.62]"
8,0.6,misma reunión,corrida 0,14,"0.20 [0.12, 0.30]","0.52 [0.34, 0.69]","0.29 [0.18, 0.38]"
9,0.6,misma reunión,corrida 1,14,"0.20 [0.12, 0.31]","0.52 [0.34, 0.69]","0.29 [0.18, 0.39]"


**Resultado.** Por corrida, los intervalos son anchos (por ejemplo, con τ = 0,60 y misma reunión,
la precisión es 0,20 [0,12; 0,30] y la cobertura 0,52 [0,34; 0,69]), y los de los dos ámbitos se
**solapan** en cada corrida: con 27 decisiones de referencia y 69 a 71 extraídas, la diferencia
entre ámbitos de una sola corrida no se distingue del ruido. Agrupando las tres corridas los
intervalos se angostan (cobertura 0,52 [0,41; 0,62] frente a 0,77 [0,66; 0,84] con toda la
serie, que dejan de solaparse), pero esa mejora es en parte aparente: las tres corridas
extraen de las mismas cuatro reuniones y comparten la misma referencia, de modo que no aportan
información independiente. Las tres corridas dan casi las mismas cifras entre sí (TP de 14 a 21
según el ámbito); la variación entre corridas es mucho menor que la diferencia entre ámbitos.

## 2.2 Cuántos emparejamientos cambian

Se comparan los pares (decisión extraída, decisión de referencia) que emite cada variante.
«Comunes» son los que salen en las dos; «solo misma reunión» y «solo serie» son los que salen
en una. Un par «de cruce» une una decisión extraída con una de referencia de **otra reunión**.

In [4]:
def pair_keys(matches):
    return {(m.predicted.id, m.reference.decision_id) for m in matches}


rows = []
for tau in DETAIL_TAUS:
    for i, decisions in enumerate(decisions_by_run):
        same = scored_matches(
            reference, decisions, similarity, tau=tau, alignment_scope="origin_meetings"
        )
        wide = scored_matches(reference, decisions, similarity, tau=tau, alignment_scope="series")
        ks, kw = pair_keys(same), pair_keys(wide)
        rows.append(
            {
                "τ": tau,
                "corrida": i,
                "TP misma reunión": len(same),
                "TP serie": len(wide),
                "pares comunes": len(ks & kw),
                "solo misma reunión": len(ks - kw),
                "solo serie": len(kw - ks),
                "de cruce (serie)": sum(m.crosses_meetings for m in wide),
            }
        )
changes = pd.DataFrame(rows)
changes

,τ,corrida,TP misma reunión,TP serie,pares comunes,solo misma reunión,solo serie,de cruce (serie)
0,0.5,0,21,27,9,12,18,16
1,0.5,1,21,27,12,9,15,13
2,0.5,2,21,27,11,10,16,14
3,0.6,0,14,21,9,5,12,12
4,0.6,1,14,21,12,2,9,9
5,0.6,2,14,20,10,4,10,10
6,0.7,0,3,4,3,0,1,1
7,0.7,1,4,5,4,0,1,1
8,0.7,2,3,4,3,0,1,1


**Resultado.** Con τ = 0,60, toda la serie logra 21, 21 y 20 emparejamientos por corrida frente a 14 en
la misma reunión: siete más (un 50 %). De los pares de toda la serie, entre 9 y 12 por corrida
cruzan reuniones. No es un simple agregado de pares: de los 14 pares de la misma reunión, entre
9 y 12 se conservan y entre 2 y 5 desaparecen, porque la asignación uno a uno se reorganiza
cuando se habilitan más candidatas (una extraída que antes se emparejaba en su reunión pasa a
emparejarse con otra decisión de referencia y libera la suya). Con τ = 0,50 la diferencia es
mayor (27 frente a 21 por corrida, con 13 a 16 cruces) y con τ = 0,70 es de un par por corrida.

## 2.3 Casos concretos de emparejamientos que cruzan reuniones

¿Son legítimos (la misma decisión que se retoma más tarde) o espurios? Como ES2015 es una serie
de desarrollo, es admisible leer el contenido ([ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md)).
La referencia está redactada en español y la extracción en inglés; el modelo de embeddings
es multilingüe. Primero todos los cruces de la corrida 0 con τ = 0,60, de mayor a menor
similitud.

In [5]:
TAU_EX = 0.60
wide0 = scored_matches(
    reference, decisions_by_run[0], similarity, tau=TAU_EX, alignment_scope="series"
)
crossing = sorted((m for m in wide0 if m.crosses_meetings), key=lambda m: -m.similarity)
pd.DataFrame(
    [
        {
            "sim": round(m.similarity, 3),
            "extraída (reunión)": m.predicted.meeting_id[-1],
            "extraída": m.predicted.content[:100],
            "referencia (reunión)": m.reference.meeting_id[-1],
            "referencia": m.reference.content[:100],
        }
        for m in crossing
    ]
)

,sim,extraída (reunión),extraída,referencia (reunión),referencia
0,0.714,d,People will be willing to spend twenty five Euros on this product,a,fijar el precio de venta en 25 euros
1,0.681,c,The product will feature photos of fruit or be banana-shaped.,d,"los esquemas de color se nombran y colorean según frutas y verduras (p. ej., banana negro y amarillo"
2,0.659,c,"The team decided that the buttons would light up from the back with an internal light source, requir",b,que se incorpore luz a los botones
3,0.654,d,our operating cost is twelve twenty Euros,a,"mantener el costo de producción en un máximo absoluto de 12,50 euros"
4,0.652,c,The team agreed that the buttons should be black to avoid colour clashing and to allow for a simplis,d,Tener 2 colores uno para los botones y otro para las carcasas
5,0.649,d,It's easy to use,c,usar un chip regular
6,0.639,d,The remote control casing will be interchangeable and the color scheme will be based on different fr,c,incorporar carcasas intercambiables al control remoto
7,0.637,b,"The target group will be international, with a focus on North America and the UK.",a,dirigir el control remoto al mercado internacional
8,0.637,c,The remote control should be technologically innovative,a,control remoto para la television
9,0.633,c,The actual remote would be hard plastic and the casings rubber,d,control de plastico y goma


Tres de ellos, elegidos por su texto para ilustrar los desenlaces posibles (la lectura es
cualitativa, de este análisis, y no se midió qué fracción de los cruces es de cada tipo):

In [6]:
def find(fragment):
    return next(m for m in crossing if fragment in m.predicted.content)


for fragment in ("twenty five Euros", "interchangeable plates", "It's easy to use"):
    m = find(fragment)
    route = f"{m.predicted.meeting_id[-1]} -> {m.reference.meeting_id[-1]}"
    print(f"[{route}] similitud {m.similarity:.3f}")
    print(f"  extraída  : {m.predicted.content}")
    print(f"  referencia: {m.reference.content}")
    print()

[d -> a] similitud 0.714
  extraída  : People will be willing to spend twenty five Euros on this product
  referencia: fijar el precio de venta en 25 euros

[c -> d] similitud 0.627
  extraída  : The participants decided to go with interchangeable plates for the remote control case
  referencia: tener carcasas intercambiables

[d -> c] similitud 0.649
  extraída  : It's easy to use
  referencia: usar un chip regular



**Lectura.** Los tres casos ilustran desenlaces distintos:

- *Precio de 25 euros* (extraída en la reunión d, referencia de la reunión a, similitud 0,714):
  es el mismo asunto, el precio de venta, retomado en la última reunión. Re-enunciado plausible.
  Matiz: la frase extraída, «la gente estará dispuesta a gastar 25 euros», es más una
  valoración que un acuerdo.
- *Carcasas intercambiables* (extraída en c, referencia de d): el mismo acuerdo se discute en c
  y en d y existe una decisión de referencia en cada reunión; la asignación global cruza
  las parejas (la extraída de d se empareja con la de referencia de c, tabla anterior, y esta con la
  de d). El contenido coincide: es un cruce sin error de fondo, pero que solo aparece porque
  las dos reuniones tratan lo mismo.
- *«Es fácil de usar»* frente a *«usar un chip regular»* (similitud 0,649): comparten apenas la
  palabra *usar*; es un emparejamiento **espurio**.

De los 12 cruces de la corrida 0 con τ = 0,60, una lectura cualitativa (un solo lector, sin
medir acuerdo) ubica unos 8 como el mismo asunto retomado en otra reunión (precio, esquema de
frutas, luces en los botones, mercado internacional, plástico y goma, carcasas) y unos 4 como
débiles o espurios (el chip, «Kinetic Energy», «control remoto para la televisión» y, en menor
grado, el costo de producción de 12,20 frente a un máximo de 12,50). Hay entonces **ambas
cosas**: la serie desarrolla un mismo producto en cuatro reuniones, así que re-enunciar es
esperable, pero un tercio de los cruces leídos no es el mismo asunto. Queda abierta una
pregunta de fondo: una decisión que se retoma en una reunión posterior, ¿es una extracción
correcta (existe en la referencia) o una repetición que debería contar como falso positivo en
esa reunión?

Para dimensionar: si los cruces fueran en general re-enunciados legítimos, su similitud debería
ser comparable a la de los emparejamientos de la misma reunión.

In [7]:
rows = []
for tau in DETAIL_TAUS:
    sims = {True: [], False: []}
    for decisions in decisions_by_run:
        for m in scored_matches(
            reference, decisions, similarity, tau=tau, alignment_scope="series"
        ):
            sims[m.crosses_meetings].append(m.similarity)
    for crosses, label in ((False, "misma reunión"), (True, "cruce")):
        v = sims[crosses]
        rows.append(
            {
                "τ": tau,
                "tipo de par (ámbito serie)": label,
                "n": len(v),
                "similitud mediana": round(float(np.median(v)), 3) if v else None,
            }
        )
pd.DataFrame(rows)

,τ,tipo de par (ámbito serie),n,similitud mediana
0,0.5,misma reunión,38,0.664
1,0.5,cruce,43,0.633
2,0.6,misma reunión,31,0.681
3,0.6,cruce,31,0.649
4,0.7,misma reunión,10,0.731
5,0.7,cruce,3,0.714


**Resultado.** La similitud mediana de los cruces es algo menor que la de los pares de la misma
reunión (0,65 frente a 0,68 con τ = 0,60; 0,63 frente a 0,66 con τ = 0,50), pero la diferencia
es pequeña: la similitud por sí sola no separa los cruces legítimos de los espurios. Con
τ = 0,70 quedan solo 3 cruces en las tres corridas.

---

# 3. Rol de las decisiones con evidencia de varios hablantes

Se cuentan las decisiones de referencia por cantidad de roles en su evidencia y luego la
cobertura por rol con las dos reglas, con τ = 0,60 y, para cada una, los dos ámbitos. Con la
regla **mayoritaria** cada decisión cuenta una vez, en un solo rol; con **cada rol** cuenta una
vez en cada rol involucrado, de modo que los denominadores suman más de 27.

In [8]:
multi = [r for r in reference if len(r.roles) > 1]
print(f"Decisiones de referencia con evidencia de más de un rol: {len(multi)} de {len(reference)}")
print(
    pd.Series([len(r.roles) for r in reference])
    .value_counts()
    .sort_index()
    .rename("n decisiones por cantidad de roles")
    .to_frame()
    .T.to_string()
)
pd.DataFrame(
    [
        {"decisión": r.decision_id, "rol mayoritario": r.role, "roles": ", ".join(r.roles)}
        for r in multi
    ]
)

Decisiones de referencia con evidencia de más de un rol: 6 de 27
                                     1  2  3
n decisiones por cantidad de roles  21  5  1


,decisión,rol mayoritario,roles
0,ES2015b.d01,ME,"ME, UI, PM"
1,ES2015b.d08,ME,"ME, PM"
2,ES2015c.d02,PM,"PM, ME"
3,ES2015c.d03,ME,"ME, PM"
4,ES2015c.d07,ME,"ME, PM"
5,ES2015c.d08,ID,"UI, ID"


In [9]:
def role_table(tau, scope):
    p = pooled(
        tau,
        scope,
        predicted_role_of=evidence_role_resolver(),
        predicted_roles_of=evidence_roles_resolver(),
    )
    roles = sorted(set(p.by_role) | set(p.by_role_every_role))
    rows = []
    for role in roles:
        a, b = p.by_role.get(role), p.by_role_every_role.get(role)
        rows.append(
            {
                "rol": role,
                "ref. (mayoritario)": a.n_reference // len(runs) if a else 0,
                "cobertura (mayoritario)": a.recall if a and a.n_reference else None,
                "ref. (cada rol)": b.n_reference // len(runs) if b else 0,
                "cobertura (cada rol)": b.recall if b and b.n_reference else None,
                "precisión (mayoritario)": a.precision if a else None,
                "precisión (cada rol)": b.precision if b else None,
            }
        )
    return pd.DataFrame(rows).set_index("rol").round(3)


role_tables = {}
for label, scope in SCOPES.items():
    role_tables[label] = role_table(0.60, scope)
    print(f"τ = 0,60, ámbito {label} (tres corridas agrupadas; «ref.» es por corrida)")
    display(role_tables[label])

τ = 0,60, ámbito misma reunión (tres corridas agrupadas; «ref.» es por corrida)


,ref. (mayoritario),cobertura (mayoritario),ref. (cada rol),cobertura (cada rol),precisión (mayoritario),precisión (cada rol)
rol,,,,,,
ID,5,0.600,5,0.600,0.240,0.240
ME,6,0.500,7,0.571,0.136,0.130
PM,15,0.467,19,0.526,0.227,0.213
UI,1,1.000,3,1.000,0.148,0.125


τ = 0,60, ámbito toda la serie (tres corridas agrupadas; «ref.» es por corrida)


,ref. (mayoritario),cobertura (mayoritario),ref. (cada rol),cobertura (cada rol),precisión (mayoritario),precisión (cada rol)
rol,,,,,,
ID,5,0.933,5,0.933,0.260,0.260
ME,6,0.500,7,0.571,0.273,0.261
PM,15,0.800,19,0.789,0.375,0.383
UI,1,1.000,3,1.000,0.148,0.125


**Resultado.** Con el cargador de referencia actual, **6 de las 27** decisiones tienen evidencia
de más de un rol (21 con un rol, 5 con dos y 1 con tres), no las 9 que menciona el issue #99:
la diferencia se anota como pregunta abierta, no se resuelve aquí. Contar la decisión en cada
rol sube los denominadores de 27 a 34 (el rol de interfaz de usuario pasa de 1 a 3
decisiones) y mueve la cobertura por rol poco: con τ = 0,60 y misma reunión, ME pasa de 0,50
a 0,57, PM de 0,47 a 0,53, y ID y UI no cambian; con toda la serie los cambios son de −0,01
a +0,07. Las diferencias son de menos de 0,08 en cobertura y los roles tienen entre 1 y 19
decisiones de referencia por corrida, de modo que están dentro del ruido que muestran los
intervalos de la sección 2.1. La regla de roles **no cambia la lectura general** en estos
datos; lo que sí cambia es la cantidad de decisiones que pesa cada rol. La precisión por
rol varía aún menos (menos de 0,03).

---

# 4. Intervalos: independientes frente a agrupados por reunión

`pool_scores` calcula como intervalos de referencia Wilson (precisión, cobertura) y *bootstrap* de F1 remuestreando
**decisiones** como si fueran independientes. Pero las decisiones de una misma reunión comparten
contexto (el mismo tema, los mismos hablantes, la misma calidad de transcripción), y las tres
corridas extraen de las mismas reuniones. Aquí se compara con un *bootstrap* que remuestrea
**reuniones completas** (`meeting_cluster_intervals`): en cada remuestra se sortean cuatro
reuniones con reposición, y se suman las decisiones extraídas y de referencia de las reuniones
sorteadas. Las tres corridas se agrupan dentro de cada reunión.

Con **cuatro** reuniones el intervalo por conglomerados es muy grueso (hay solo 35 remuestras
distintas posibles); sirve para ver el orden de magnitud de lo que ocultan los intervalos
independientes, no como un intervalo definitivo.

In [10]:
def hit_lists(tau, scope):
    pred, ref_ = [], []
    for decisions in decisions_by_run:
        accepted = [d for d in decisions if d.is_decision]
        s = score_series(SERIES, reference, decisions, similarity, tau=tau, alignment_scope=scope)
        pred += [(d.meeting_id, h) for d, h in zip(accepted, s.predicted_hits, strict=True)]
        ref_ += [(r.meeting_id, h) for r, h in zip(reference, s.reference_hits, strict=True)]
    return pred, ref_


rows = []
for tau in DETAIL_TAUS:
    for label, scope in SCOPES.items():
        p = pooled(tau, scope)
        c = meeting_cluster_intervals(*hit_lists(tau, scope), settings)
        for metric, std, clu, point in (
            ("precisión", p.precision_interval_reference, c.precision_interval, p.precision),
            ("cobertura", p.recall_interval_reference, c.recall_interval, p.recall),
            ("F1", p.f1_interval_reference, c.f1_interval, p.f1),
        ):
            rows.append(
                {
                    "τ": tau,
                    "ámbito": label,
                    "métrica": metric,
                    "valor": round(point, 3),
                    "estándar": f"[{std[0]:.2f}, {std[1]:.2f}]",
                    "ancho estándar": round(std[1] - std[0], 3),
                    "por reunión": f"[{clu[0]:.2f}, {clu[1]:.2f}]",
                    "ancho por reunión": round(clu[1] - clu[0], 3),
                }
            )
intervals = pd.DataFrame(rows)
intervals

,τ,ámbito,métrica,valor,estándar,ancho estándar,por reunión,ancho por reunión
0,0.5,misma reunión,precisión,0.301,"[0.24, 0.37]",0.124,"[0.22, 0.33]",0.111
1,0.5,misma reunión,cobertura,0.778,"[0.68, 0.85]",0.179,"[0.37, 1.00]",0.632
2,0.5,misma reunión,F1,0.434,"[0.37, 0.50]",0.133,"[0.28, 0.50]",0.220
3,0.5,toda la serie,precisión,0.388,"[0.32, 0.46]",0.131,"[0.28, 0.46]",0.180
4,0.5,toda la serie,cobertura,1.000,"[0.95, 1.00]",0.045,"[1.00, 1.00]",0.000
5,0.5,toda la serie,F1,0.559,"[0.49, 0.62]",0.134,"[0.44, 0.63]",0.192
6,0.6,misma reunión,precisión,0.201,"[0.15, 0.26]",0.108,"[0.13, 0.23]",0.100
7,0.6,misma reunión,cobertura,0.519,"[0.41, 0.62]",0.213,"[0.21, 0.69]",0.477
8,0.6,misma reunión,F1,0.290,"[0.23, 0.35]",0.121,"[0.16, 0.34]",0.184
9,0.6,toda la serie,precisión,0.297,"[0.24, 0.36]",0.123,"[0.17, 0.39]",0.222


In [11]:
# Share of per-decision "widths" vs meeting-cluster "widths", by metric.
ratio = intervals.assign(
    razón=(intervals["ancho por reunión"] / intervals["ancho estándar"]).round(2)
)
ratio.groupby("métrica")["razón"].agg(["min", "median", "max"])

,min,median,max
métrica,,,
F1,0.97,1.585,2.01
cobertura,0.00,1.865,3.53
precisión,0.52,1.100,1.80


**Resultado.** Con τ = 0,60, el intervalo por reuniones es más ancho que el estándar casi siempre:

| ámbito | métrica | ancho estándar | ancho por reunión | razón |
|---|---|---|---|---|
| misma reunión | precisión | 0,108 | 0,100 | 0,9 |
| misma reunión | cobertura | 0,213 | 0,477 | 2,2 |
| misma reunión | F1 | 0,121 | 0,184 | 1,5 |
| toda la serie | precisión | 0,123 | 0,222 | 1,8 |
| toda la serie | cobertura | 0,182 | 0,429 | 2,4 |
| toda la serie | F1 | 0,134 | 0,269 | 2,0 |

La cobertura es la métrica más afectada: las reuniones aportan entre 4 y 8 decisiones de
referencia y varían mucho en lo bien que se extraen, algo que el intervalo de Wilson no ve. La
precisión de la misma reunión es la excepción (mismo ancho), porque hay muchas decisiones
extraídas por reunión. En el conjunto de τ y ámbitos mostrados, la razón mediana es 1,1 para
precisión, 1,6 para F1 y 1,9 para cobertura. En algunos casos la razón es 0 porque el
intervalo por reuniones se degenera (con toda la serie y τ = 0,50 la cobertura vale 1,00 en
todas las remuestras). **Advertencias**: con cuatro reuniones el *bootstrap* por conglomerados
es burdo (solo 35 remuestras distintas) y un intervalo calculado con tan pocos conglomerados
suele ser, a su vez, algo estrecho; y los intervalos estándar de las cifras agrupadas están
infladamente angostos por contar tres veces las mismas reuniones. El orden de magnitud sí es
informativo: ignorar la agrupación por reunión puede subestimar la incertidumbre de la
cobertura y de F1 en un factor de alrededor de dos.

---

# 5. Insumos para la decisión

Lo que sigue enumera las opciones y lo que muestran los datos. **No recomienda ni decide.**

**Ámbito de la alineación**

| Opción | Qué muestran los datos (ES2015, 3 corridas, desarrollo) |
|---|---|
| Misma reunión (estricta) | Cifras más bajas (con τ = 0,60: precisión 0,20, cobertura 0,52, F1 0,29). Nunca empareja una extracción con una decisión de otra reunión, así que evita los cruces espurios (unos 4 de 12 en la lectura cualitativa). En cambio, no reconoce como correcta una decisión que la extracción asigna a una reunión posterior donde se la retoma, ni cuando coincide con la referencia de otra reunión. |
| Toda la serie (permisiva) | Cifras más altas (precisión 0,30, cobertura 0,77, F1 0,43 con τ = 0,60). La mitad de sus aciertos cruza reuniones (31 de 62 agrupados); de los 12 cruces leídos, unos 8 son el mismo asunto retomado y unos 4 débiles o espurios. Ninguna de las dos variantes cambia el orden entre ámbitos a lo largo de τ. |
| Primaria en una, la otra como sensibilidad (como está hoy en el código) | El código ya calcula las dos (`score_series_views`). Declara el resultado en ambas y deja ver cuánto depende del ámbito: la diferencia es grande con τ bajos y pequeña con τ altos. |

Observaciones que no dependen de la opción: la elección de τ mueve más las cifras que el
ámbito; el modelo de embeddings da similitudes bajas entre inglés y español (entre 0,75 y 0,80
los pares dejan de alcanzar el umbral); y la pregunta de si una decisión re-enunciada en una reunión posterior es acierto o
repetición no se resuelve con estos datos.

**Rol de una decisión con evidencia de varios roles**

| Opción | Qué muestran los datos |
|---|---|
| Rol mayoritario (empate: turno más temprano) | Cada decisión cuenta una vez; 27 decisiones en total. Es simple y mantiene el denominador. |
| Contar la decisión en cada rol | 34 conteos en total; cobertura por rol que difiere en menos de 0,08 de la anterior; el rol UI pasa de 1 a 3 decisiones. Refleja que la evidencia involucra a varias personas, a costa de contar una misma decisión más de una vez. |

El cargador actual encuentra 6 decisiones con varios roles y no 9: conviene reconciliar esa cifra
antes de citarla en un ADR.

**Intervalos**

| Opción | Qué muestran los datos |
|---|---|
| Declarar la limitación (intervalos estándar, optimistas) | Con τ = 0,60, el intervalo por reuniones es entre 1,5 y 2,4 veces más ancho para cobertura y F1 (precisión de la misma reunión, sin diferencia). La declaración debería nombrar a la cobertura y a F1 como las métricas más afectadas. |
| Adoptar un intervalo por conglomerados de reuniones | Es más honesto con la estructura de los datos, pero con cuatro reuniones por serie es muy burdo; con más series de evaluación sería más estable. La función `meeting_cluster_intervals` ya existe y es opt-in. |

**Qué no sale de este análisis**: ninguna cifra de aquí es un resultado (serie de desarrollo,
OpenKOS 0.5.5 provisional, n pequeño); el análisis de la regla se hizo sobre una sola serie y
una sola configuración de extracción.

---

# 6. Decisión posterior

Las tres cuestiones que este análisis dejó abiertas se decidieron después, y quedan registradas en el
[ADR 0011](../docs/decisions/0011-adr-alineacion-rol-e-intervalos-del-experimento-a.md): el ámbito
primario es la misma reunión (reuniones de origen), con la serie completa como sensibilidad; el rol
primario es el mayoritario, con cada rol como sensibilidad; y el intervalo primario pasa a ser el
*bootstrap* por reuniones, con Wilson y el *bootstrap* por decisión como referencia. El ADR no
fija τ. Desde ese cambio, `pool_scores` entrega el intervalo por reuniones en `*_interval` y el
intervalo independiente en `*_interval_reference`; las celdas de este cuaderno que muestran el
«intervalo estándar» leen los campos de referencia, de modo que sus cifras no cambian.